# Training on the features you predicted with

In online learning the features often arrive before the label. Between the question and the answer, the transformers in a pipeline keep learning. If the model is then updated with the *arrival-time* features, it trains on a different view of the observation than the one it actually predicted with. With a running `preprocessing.StandardScaler`, a supervised `feature_extraction.TargetAgg`, or any other evolving transformer, that is a silent train/serve mismatch.

River's history handles this in two ways:

| Interface | Prediction phase | Feedback phase |
| --- | --- | --- |
| `learn_one` (default) | transform only | all unsupervised steps learn on the arrival features; supervised steps and the final predictor learn on transformed features |
| contexts (`*_with_context`) | unsupervised steps learn **then** transform; supervised steps only transform; the predictor only predicts | the predictor learns with the exact features it predicted with; supervised transformers learn with the input they transformed |

The second row is opt-in, for `compose.Pipeline`.


## The two phases


In [ ]:
from river import compose, feature_extraction, linear_model, preprocessing, stats

model = compose.Pipeline(
    compose.Select("amount", "shop"),
    compose.TransformerUnion(
        compose.Select("amount") | preprocessing.StandardScaler(),
        feature_extraction.TargetAgg(by="shop", how=stats.Mean()),
    ),
    linear_model.LinearRegression(),
)

Ask a question. The context records what each supervised step saw:

In [ ]:
prediction, context = model.predict_one_with_context({"amount": 10.0, "shop": "A"})
prediction

The scaler has already learned from this observation, but `TargetAgg` has not seen the target:

In [ ]:
scaler = model[1][0][1]
agg = model[1][1]
dict(scaler.counts), agg._groups[("A",)].n

Once the answer arrives, feed it back. `TargetAgg` learns from the input it transformed, and the regressor learns with the exact features it predicted with:

In [ ]:
model.learn_one_from_context(context, 4.0)
agg._groups[("A",)].n, agg._groups[("A",)]._mean

The scaler is not learned again, and no `transform_one` is called to rebuild the observation.

What happens, step by step:

1. `StandardScaler` (in its branch) learns from `{"amount": 10.0}` and then transforms it.
2. `TargetAgg` only transforms: it cannot see `4.0` before the answer event.
3. `LinearRegression` only predicts.
4. On feedback, `TargetAgg` learns from the input it transformed in step 2, and `LinearRegression` learns from the exact feature dictionary it predicted with.
5. The scaler is not learned again.

The old interface is untouched: `model.predict_one(x)` followed by `model.learn_one(x, y)` behaves exactly as in previous versions.


## Contexts are one-shot and belong to their pipeline

A context can be learned from once, or released without learning:

In [ ]:
prediction, context = model.predict_one_with_context({"amount": 5.0, "shop": "A"})
model.learn_one_from_context(context, 3.0)
model.discard_context(context)

- `discard_context` returns `True` the first time it releases a pending context and `False` for a context that already ended.
- A context created by another pipeline (including a clone), a forged object, or a context already learned from raises `ValueError`, and no step is updated.
- A failed feedback propagates the original exception and terminates the context: the steps that already ran are not replayed. A failed prediction returns no context at all.


## Delayed evaluation

`evaluate.progressive_val_score` and `evaluate.iter_progressive_val_score` accept `feature_view`. The default `'arrival'` preserves the previous behaviour; `'prediction'` uses the context protocol:

In [ ]:
import random

from river import evaluate, metrics

rng = random.Random(42)
dataset = [
    ({"amount": rng.uniform(0, 100), "shop": rng.choice(["A", "B", "C"])}, rng.uniform(0, 100))
    for _ in range(500)
]

model = compose.Pipeline(
    compose.Select("amount", "shop"),
    compose.TransformerUnion(
        compose.Select("amount") | preprocessing.StandardScaler(),
        feature_extraction.TargetAgg(by="shop", how=stats.Mean()),
    ),
    linear_model.LinearRegression(),
)

metric = evaluate.progressive_val_score(
    dataset=dataset,
    model=model,
    metric=metrics.MAE(),
    delay=3,
    feature_view="prediction",
)
metric

- Event ordering is still produced by `stream.simulate_qa`: the same moments, delays, same-instant ordering and end-of-stream feedback.
- The metric is updated at the answer event with the prediction that was actually issued, while feature learning happens at the question event.
- `feature_view='prediction'` requires a pipeline that supports the protocol; anything else is rejected before the dataset is consumed.
- The evaluator releases the contexts it still owns and closes the input iterator when it finishes, fails, or is closed explicitly.


## Supported shapes and rejections

The protocol supports trees built from ordinary transformers, `base.SupervisedTransformer`, nested transformation pipelines and `compose.TransformerUnion`, with a supervised regressor or classifier at the end. Mixing supervised and unsupervised branches is fine, and nesting depth is not limited. Detection relies on the `learn_one` / `transform_one` protocol, not on class names, so custom steps that follow the protocol work.

Sharing a *stateful* transformer instance between two branches is rejected up front (two branches would otherwise record and update the same state twice). So are pipelines with an anomaly filter or an active learner.


## Why the chosen design

A context stores, per observation, only two things:

- a snapshot of the input that entered each **supervised** transformer, and
- the exact feature dictionary given to the final predictor.

Unsupervised steps are never re-run, so they need no snapshot. A context is held weakly by the pipeline, and the bookkeeping container removes itself once it is empty, so a pipeline with nothing pending keeps no context table and never grows a history of finished IDs. Learned, discarded and failed contexts drop their payloads even if the caller keeps the object around.

Costs:

- **Wide dictionaries**: each pending context holds one shallow copy of the input per supervised leaf, plus the final feature dict. Cost is `O(pending × width × supervised leaves)`; it does not depend on how many observations were already seen.
- **Deep branches**: every actual leaf transforms once per prediction, so deeper trees cost more per prediction, but retention per pending observation grows only with the number of supervised leaves on the executed paths.
- **Long feedback delays**: the number of retained contexts is exactly the number of unanswered questions, so a long delay costs proportionally more memory. Discarding answered observations promptly keeps it flat.


## Verification

Commands used while developing this change (Python 3.11, repository dependencies):

```bash
python3 -m pytest river/compose/test_context.py -q
python3 -m pytest river/compose/test_context_resources.py -q -s
python3 -m pytest river/evaluate/test_feature_view.py river/evaluate -q
python3 -m pytest river/compose river/evaluate river/stream river/preprocessing \
    river/feature_extraction river/linear_model -q
python3 -m pytest river/compose/pipeline.py river/evaluate/progressive_validation.py \
    --doctest-modules -q
```

Observed at the time of writing:

- `test_context.py`: 44 passed.
- `test_context_resources.py`: 14 passed (about 26 s; includes 100k-observation runs).
    - pending=8, width=2: retained 8 contexts, peak retained payload about 6.0 KB, peak traced memory about 68 KiB.
    - pending=128, width=2: retained 128 contexts, peak retained payload about 96 KB, peak traced memory about 154 KiB.
    - pending=8, width=16: retained 8 contexts, peak retained payload about 28 KB.
    - pending=128, width=16: retained 128 contexts, peak retained payload about 449 KB, peak traced memory about 349 KiB.
    - depth=1 tree, 16 pending: peak retained payload about 13.7 KB, about 1.8 s.
    - depth=3 tree, 16 pending: peak retained payload about 38 KB, about 7 s (the deeper tree runs eight times as many leaves per prediction; retention only doubles).
- `evaluate` (including `test_feature_view.py`): 19 passed.
- Affected suites (`compose`, `evaluate`, `stream`, `preprocessing`, `feature_extraction`, `linear_model`): 240 passed, 10 warnings (all pre-existing pandas/numpy deprecation warnings from `linear_model` mini-batch tests).
- Doctests for `compose/pipeline.py` and `evaluate/progressive_validation.py`: 6 passed.

Paths not exercised here:

- Mini-batch (`learn_many` / `predict_many`) context variants, which this version does not provide.
- Anomaly filters and active learners with `feature_view='prediction'`; they are deliberately rejected.
- `compose.TransformerProduct` and other union subclasses with a custom merge; they are rejected explicitly.
- An end-to-end run of the documentation notebooks (`make doc`), which also needs the MkDocs toolchain.
